In [1]:

import pandas as pd
import numpy as np

np.random.seed(42)

n = 1000

products = ["Laptop", "Phone", "Headphones", "Keyboard", "Mouse"]
categories = {
    "Laptop": "Electronics",
    "Phone": "Electronics",
    "Headphones": "Accessories",
    "Keyboard": "Accessories",
    "Mouse": "Accessories"
}
prices = {
    "Laptop": 50000,
    "Phone": 25000,
    "Headphones": 2000,
    "Keyboard": 1500,
    "Mouse": 800
}

df = pd.DataFrame({
    "OrderID": range(1001, 1001 + n),
    "OrderDate": pd.to_datetime(
        np.random.choice(
            pd.date_range("2025-01-01", "2025-12-31"),
            n
        )
    ),
    "Product": np.random.choice(products, n),
    "Region": np.random.choice(
        ["North", "South", "East", "West"], n
    ),
    "Quantity": np.random.randint(1, 6, n)
})

df["Category"] = df["Product"].map(categories)
df["UnitPrice"] = df["Product"].map(prices)
df["UnitCost"] = df["UnitPrice"] * 0.70
df["Sales"] = df["Quantity"] * df["UnitPrice"]
df["Profit"] = df["Quantity"] * (
    df["UnitPrice"] - df["UnitCost"]
)

df.to_csv("retail_sales.csv", index=False)

print("Dataset created successfully!")
print(df.head())
print("Dataset shape:", df.shape)


Dataset created successfully!
   OrderID  OrderDate     Product Region  Quantity     Category  UnitPrice  \
0     1001 2025-04-13  Headphones  North         1  Accessories       2000   
1     1002 2025-12-15  Headphones  North         4  Accessories       2000   
2     1003 2025-09-28      Laptop  North         1  Electronics      50000   
3     1004 2025-04-17       Phone  North         4  Electronics      25000   
4     1005 2025-03-13       Phone   West         4  Electronics      25000   

   UnitCost   Sales   Profit  
0    1400.0    2000    600.0  
1    1400.0    8000   2400.0  
2   35000.0   50000  15000.0  
3   17500.0  100000  30000.0  
4   17500.0  100000  30000.0  
Dataset shape: (1000, 10)


In [2]:

import pandas as pd
import sqlite3

# 1. Read the CSV dataset
df = pd.read_csv("retail_sales.csv")

# 2. Connect to the SQLite database
conn = sqlite3.connect("retail_sales.db")

# 3. Store the data in a database table
df.to_sql("sales", conn, if_exists="replace", index=False)

print("Database created successfully!")

# 4. Total revenue and profit
query1 = """
SELECT
    SUM(Sales) AS Total_Revenue,
    SUM(Profit) AS Total_Profit
FROM sales;
"""
print("\nTotal Revenue and Profit:")
print(pd.read_sql_query(query1, conn))

# 5. Best-selling products
query2 = """
SELECT Product, SUM(Sales) AS Total_Sales
FROM sales
GROUP BY Product
ORDER BY Total_Sales DESC;
"""
print("\nSales by Product:")
print(pd.read_sql_query(query2, conn))

# 6. Sales by region
query3 = """
SELECT Region, SUM(Sales) AS Total_Sales
FROM sales
GROUP BY Region
ORDER BY Total_Sales DESC;
"""
print("\nSales by Region:")
print(pd.read_sql_query(query3, conn))

# 7. Monthly sales
query4 = """
SELECT
    strftime('%Y-%m', OrderDate) AS Month,
    SUM(Sales) AS Monthly_Sales
FROM sales
GROUP BY Month
ORDER BY Month;
"""
print("\nMonthly Sales:")
print(pd.read_sql_query(query4, conn))

# 8. Profit by category
query5 = """
SELECT Category, SUM(Profit) AS Total_Profit
FROM sales
GROUP BY Category
ORDER BY Total_Profit DESC;
"""
print("\nProfit by Category:")
print(pd.read_sql_query(query5, conn))

# 9. Close database connection
conn.close()
print("\nAnalysis completed!")


Database created successfully!

Total Revenue and Profit:
   Total_Revenue  Total_Profit
0       46958700    14087610.0

Sales by Product:
      Product  Total_Sales
0      Laptop     29100000
1       Phone     15250000
2  Headphones      1238000
3    Keyboard       871500
4       Mouse       499200

Sales by Region:
  Region  Total_Sales
0  North     12798400
1  South     12148600
2   West     11185400
3   East     10826300

Monthly Sales:
      Month  Monthly_Sales
0   2025-01        3721800
1   2025-02        4054800
2   2025-03        3186800
3   2025-04        4942000
4   2025-05        4377100
5   2025-06        3738500
6   2025-07        2851200
7   2025-08        4548600
8   2025-09        3117700
9   2025-10        3389400
10  2025-11        4343400
11  2025-12        4687400

Profit by Category:
      Category  Total_Profit
0  Electronics    13305000.0
1  Accessories      782610.0

Analysis completed!
